# Step 6 - Model selection (base learners individuales)

Evalua cada base learner por separado en val 2023 para confirmar que aportan al ensemble.


In [ ]:
# Imports y configuracion del proyecto
from project_config import init_notebook
config = init_notebook()

import utils
logger = utils.init_logger('info')


In [ ]:
import json
import pandas as pd
from pathlib import Path
from analytics import df_split, threshold_iter, choose_threshold_by_strategy
from models.stacking.base_learners import LGBMBaseLearner, XGBBaseLearner, CatBoostBaseLearner, RFBaseLearner

processed = Path(config['project_folder']) / config['data']['processed']['local_path']
train = pd.read_parquet(processed / 'train.parquet')
val = pd.read_parquet(processed / 'val.parquet')
selected = json.loads((processed / 'selected_features.json').read_text())

target = config['model']['objective_column']
X_tr, y_tr = df_split(train[selected + [target]], target)
X_va, y_va = df_split(val[selected + [target]], target)


## 6.1 Entrenar y comparar cada base learner


In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score
results = []
for cls in [LGBMBaseLearner, XGBBaseLearner, CatBoostBaseLearner, RFBaseLearner]:
    learner = cls()
    learner.fit(X_tr, y_tr, X_va, y_va)
    proba = learner.predict_proba(X_va)[:, 1]
    results.append({
        'model': learner.name,
        'pr_auc': average_precision_score(y_va, proba),
        'roc_auc': roc_auc_score(y_va, proba),
    })
pd.DataFrame(results)
